# Bayesian Model Comparison: WAIC, LOO-CV, and Bayes Factors

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain **why model comparison is necessary** and how the Bayesian approach differs from frequentist AIC/BIC.
2. Define the **expected log pointwise predictive density (ELPD)** and explain why it is the fundamental quantity for predictive model comparison.
3. Compute and interpret **WAIC** (Widely Applicable Information Criterion) using ArviZ.
4. Compute and interpret **PSIS-LOO** (Pareto Smoothed Importance Sampling Leave-One-Out) cross-validation and its Pareto $\hat{k}$ diagnostic.
5. Use `az.compare()` and `az.plot_compare()` to rank competing models.
6. Explain **Bayes factors**, their interpretation, and their practical limitations.
7. Perform a complete model comparison workflow: fit multiple Bayesian regression models, compare them, and validate the selection with posterior predictive checks.

## Prerequisites

- [01_bayesian_linear_regression.ipynb](01_bayesian_linear_regression.ipynb) -- Bayesian regression in PyMC
- [02_posterior_predictive.ipynb](02_posterior_predictive.ipynb) -- Posterior predictive distributions
- [Module 06: Model Selection](../06_linear_models/05_model_selection.ipynb) -- AIC, BIC, cross-validation (frequentist)
- [Module 07: MCMC Diagnostics](../07_bayesian_inference/05_mcmc_diagnostics.ipynb) -- Convergence checks

In [ ]:
import sys, os, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.system(
        "pip install -q pymc arviz statsmodels"
    )

_miktex_bin = Path.home() / "AppData/Local/Programs/MiKTeX/miktex/bin/x64"
if _miktex_bin.exists() and str(_miktex_bin) not in os.environ.get("PATH", ""):
    os.environ["PATH"] += os.pathsep + str(_miktex_bin)

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

try:
# PyTensor Windows fix: skip C compilation
os.environ["PYTENSOR_FLAGS"] = "device=cpu,floatX=float64,cxx="
    import pymc as pm
    import arviz as az
    print(f"PyMC {pm.__version__}, ArviZ {az.__version__}")
except Exception:
    pm, az = None, None
    print("PyMC/ArviZ not available -- PyMC cells will be skipped.")

sys.path.insert(0, os.path.abspath("../../src"))
from amstats.plotting import apply_style

apply_style()


class Cfg:
    root = Path("../../").resolve()
    gif_dir = root / "media" / "gifs"
    has_latex: bool = (
        shutil.which("latex") is not None or shutil.which("pdflatex") is not None
    )

    def __init__(self):
        self.gif_dir.mkdir(parents=True, exist_ok=True)


cfg = Cfg()
rng = np.random.default_rng(42)

if az is not None:
    az.style.use("arviz-darkgrid")

---

## 1. The Model Comparison Problem

In any data analysis, we face a fundamental question: **given several candidate models, which one best explains the data and will best predict future observations?**

In the frequentist world (Module 06), we addressed this with:
- **AIC** ($= -2\ell + 2p$) -- estimates expected out-of-sample KL divergence
- **BIC** ($= -2\ell + p \ln n$) -- approximates the log marginal likelihood
- **Cross-validation** -- directly estimates prediction error by holding out data

These tools work well for maximum-likelihood fits. But in the Bayesian framework, we have a **full posterior distribution** over parameters rather than a single point estimate. This changes the model comparison landscape:

| Aspect | Frequentist | Bayesian |
|--------|------------|----------|
| Parameter estimate | Single point $\hat{\theta}_{\text{MLE}}$ | Full posterior $p(\theta \mid y)$ |
| "Number of parameters" | Fixed count $p$ | Effective count (data-dependent) |
| Predictions | Plug-in: $p(\tilde{y} \mid \hat{\theta})$ | Averaged: $p(\tilde{y} \mid y) = \int p(\tilde{y} \mid \theta)\, p(\theta \mid y)\, d\theta$ |
| Model comparison tools | AIC, BIC, $k$-fold CV | **WAIC, LOO-CV, Bayes factors** |

The Bayesian approach to model comparison uses the posterior predictive distribution directly. The key insight is that a good model should **predict held-out data well**, and we can estimate this predictive ability from the posterior samples we already have -- no refitting required.

In this notebook, we focus on the **practical toolkit**: WAIC and LOO-CV via ArviZ, with a brief discussion of Bayes factors. We will *not* reimplement these from scratch -- the algorithms are numerically delicate, and ArviZ's implementations are well-tested. Instead, we explain the theory, show how to use the tools, and interpret the results.

---

## 2. Expected Log Pointwise Predictive Density (ELPD)

Before diving into WAIC and LOO, we need to understand the quantity they both estimate: the **expected log pointwise predictive density (ELPD)**.

### The setup

Suppose we have observed data $y = (y_1, \dots, y_n)$ and a model $M$ with posterior $p(\theta \mid y, M)$. For a new observation $\tilde{y}_i$ drawn from the true data-generating process $f$, the **posterior predictive density** at that point is:

$$p(\tilde{y}_i \mid y) = \int p(\tilde{y}_i \mid \theta)\, p(\theta \mid y)\, d\theta$$

This is the model's prediction for $\tilde{y}_i$, averaged over all plausible parameter values.

### ELPD definition

The **ELPD** is the expected value of the log predictive density over the true data-generating distribution:

$$\text{ELPD} = \sum_{i=1}^{n} \int f(\tilde{y}_i) \log p(\tilde{y}_i \mid y)\, d\tilde{y}_i$$

In words: for each data point, compute how well the model predicts it (in log probability), then average over what future data could look like under the true distribution $f$.

### Why ELPD?

- **Higher ELPD = better predictive accuracy.** A model that assigns high probability to observations that actually occur has a high ELPD.
- ELPD is directly related to the **Kullback-Leibler divergence** between the true distribution and the model's predictive distribution. Maximising ELPD is equivalent to minimising this KL divergence.
- The connection to AIC: Akaike showed that $\text{AIC} \approx -2 \cdot \widehat{\text{ELPD}}$ (plus a constant). So AIC is, in a sense, a frequentist approximation to ELPD.

### The problem

We cannot compute ELPD exactly because we do not know the true data-generating process $f$. We can only **estimate** it from the data we have. The two main estimation strategies are:

1. **WAIC** -- uses the within-sample log predictive density with a bias correction
2. **LOO-CV** -- estimates out-of-sample prediction by leave-one-out cross-validation

Both produce an estimate of ELPD (or equivalently, $-2 \times \text{ELPD}$, on the deviance scale). Let us examine each in turn.

---

## 3. WAIC (Widely Applicable Information Criterion)

WAIC, introduced by Watanabe (2010), is a **fully Bayesian** generalisation of AIC. Unlike AIC, which relies on the MLE and a fixed parameter count, WAIC works directly with the posterior distribution.

### Components

**Log Pointwise Predictive Density (lppd):**

$$\text{lppd} = \sum_{i=1}^{n} \log \left( \frac{1}{S} \sum_{s=1}^{S} p(y_i \mid \theta^{(s)}) \right)$$

where $\theta^{(1)}, \dots, \theta^{(S)}$ are posterior draws. For each observation $y_i$, we compute the likelihood under every posterior sample, average them, and take the log. This is the model's in-sample predictive fit.

**Effective number of parameters ($p_{\text{WAIC}}$):**

$$p_{\text{WAIC}} = \sum_{i=1}^{n} \text{Var}_{s=1}^{S} \left[ \log p(y_i \mid \theta^{(s)}) \right]$$

This is the sum, over observations, of the posterior variance of the log-likelihood. Intuitively, if the posterior is very concentrated (strong data signal), the variance of the log-likelihood is low and $p_{\text{WAIC}}$ is close to the actual parameter count. If the posterior is diffuse (weak data), $p_{\text{WAIC}}$ can be smaller, reflecting the regularisation from the prior.

### WAIC formula

$$\boxed{\text{WAIC} = -2 \left( \text{lppd} - p_{\text{WAIC}} \right)}$$

Or equivalently, as an estimate of ELPD:

$$\widehat{\text{ELPD}}_{\text{WAIC}} = \text{lppd} - p_{\text{WAIC}}$$

### Key properties

- **Computed from posterior samples** -- no refitting required. You just need the log-likelihood evaluated at each posterior draw for each observation.
- **Asymptotically equivalent to LOO-CV** (under regularity conditions).
- **Lower WAIC is better** (like AIC). Equivalently, higher $\widehat{\text{ELPD}}_{\text{WAIC}}$ is better.
- Requires that the model stores the **pointwise log-likelihood** in the InferenceData object (PyMC does this automatically with `idata_kwargs={"log_likelihood": True}`).

### Comparison with AIC

| | AIC | WAIC |
|---|---|---|
| Based on | MLE point estimate | Full posterior |
| Complexity penalty | $2p$ (fixed) | $p_{\text{WAIC}}$ (data-adaptive) |
| Requires | Log-likelihood at MLE | Pointwise log-likelihood at all posterior draws |
| Works for | Regular models | Singular models too (mixtures, etc.) |

---

## 4. LOO-CV (Leave-One-Out Cross-Validation)

### The idea

Leave-one-out cross-validation estimates the ELPD by predicting each observation from a model fit **without that observation**:

$$\widehat{\text{ELPD}}_{\text{LOO}} = \sum_{i=1}^{n} \log p(y_i \mid y_{-i})$$

where

$$p(y_i \mid y_{-i}) = \int p(y_i \mid \theta)\, p(\theta \mid y_{-i})\, d\theta$$

is the posterior predictive density for observation $i$ when that observation is left out of the fit.

### The computational problem

Naively, LOO-CV requires refitting the model $n$ times (once for each left-out observation). For Bayesian models with MCMC, this is prohibitively expensive.

### PSIS-LOO: Pareto Smoothed Importance Sampling

Vehtari, Gelman, and Gabry (2017) developed **PSIS-LOO**, a clever approximation that avoids refitting entirely. The key insight: the leave-one-out posterior $p(\theta \mid y_{-i})$ is close to the full posterior $p(\theta \mid y)$, and we can use **importance sampling** to reweight the existing posterior draws.

The importance weights are:

$$w_i^{(s)} = \frac{1}{p(y_i \mid \theta^{(s)})}$$

Intuitively, to approximate the posterior without observation $i$, we downweight draws where observation $i$ is well-explained (because in the leave-one-out posterior, that observation's influence is removed).

Raw importance weights can be highly variable, leading to unstable estimates. PSIS stabilises them by fitting a **generalised Pareto distribution** to the upper tail of the weights and replacing extreme values with the fitted distribution's quantiles.

### The Pareto $\hat{k}$ diagnostic

The shape parameter $\hat{k}$ of the fitted Pareto distribution serves as a **diagnostic** for each observation:

| $\hat{k}$ | Interpretation |
|-----------|----------------|
| $< 0.5$ | **Good.** The importance sampling approximation is reliable. |
| $0.5 - 0.7$ | **OK.** The approximation is reasonable but not great. |
| $0.7 - 1.0$ | **Problematic.** The estimate for this observation is unreliable. Consider moment matching or refitting. |
| $> 1.0$ | **Very bad.** The importance weights have infinite variance. This observation is highly influential and the LOO estimate is not trustworthy. |

High $\hat{k}$ values typically indicate **influential observations** -- points that strongly affect the posterior. These are the very points where LOO matters most, so high $\hat{k}$ is a serious warning.

### In practice

ArviZ computes PSIS-LOO via `az.loo()`. It returns the ELPD estimate, its standard error, and the pointwise Pareto $\hat{k}$ values. If you see $\hat{k} > 0.7$ for many observations, the LOO estimate is unreliable and you should:
1. Try moment matching (`az.loo(..., pointwise=True)` and inspect)
2. Refit the model without the problematic observations
3. Consider whether your model is misspecified for those data points

---

## 5. Setting Up the Worked Example

To demonstrate model comparison in practice, we simulate data from a **quadratic relationship** and then fit four competing Bayesian regression models of increasing complexity: linear, quadratic, cubic, and degree-5.

The true data-generating process:

$$y_i = 2.0 + 0.8\, x_i - 1.2\, x_i^2 + \varepsilon_i, \qquad \varepsilon_i \sim \mathcal{N}(0, 1.5^2)$$

A good model comparison procedure should identify the quadratic model as the best (or near-best) choice.

In [ ]:
# --- Simulate data from a quadratic relationship ---
n = 80
x = rng.uniform(-3, 3, n)
x_sorted = np.sort(x)

# True coefficients
TRUE_BETA = [2.0, 0.8, -1.2]  # intercept, linear, quadratic
TRUE_SIGMA = 1.5

def true_function(t):
    return TRUE_BETA[0] + TRUE_BETA[1] * t + TRUE_BETA[2] * t**2

y = true_function(x) + rng.normal(0, TRUE_SIGMA, n)

# Dense grid for plotting
x_grid = np.linspace(-3.3, 3.3, 200)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(x, y, s=30, alpha=0.6, zorder=5, label="Observed data")
ax.plot(x_grid, true_function(x_grid), "k--", linewidth=2, label="True quadratic")
ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
ax.set_title("Simulated Data: Quadratic Relationship with Gaussian Noise")
ax.legend()
plt.tight_layout()
plt.show()

print(f"n = {n}, true model: y = {TRUE_BETA[0]} + {TRUE_BETA[1]}x + ({TRUE_BETA[2]})x² + N(0, {TRUE_SIGMA}²)")

---

## 6. Fitting Four Competing Models in PyMC

We fit polynomial regression models of degree 1, 2, 3, and 5. Each model uses weakly informative priors:

- Coefficients: $\beta_j \sim \mathcal{N}(0, 10)$
- Noise: $\sigma \sim \text{HalfNormal}(10)$

We standardise the polynomial features (center and scale) to improve sampling geometry. This is good practice for polynomial regression -- raw powers of $x$ can span many orders of magnitude.

In [ ]:
def build_design_matrix(x, degree):
    """Build standardised polynomial design matrix."""
    X = np.column_stack([x**j for j in range(1, degree + 1)])
    # Standardise each column
    means = X.mean(axis=0)
    stds = X.std(axis=0)
    stds[stds == 0] = 1.0  # avoid division by zero
    X_std = (X - means) / stds
    return X_std, means, stds


def fit_polynomial_model(x, y, degree, model_name, random_seed=42):
    """
    Fit a Bayesian polynomial regression of given degree.

    Returns the PyMC model, InferenceData, and design matrix info.
    """
    X_std, means, stds = build_design_matrix(x, degree)

    with pm.Model() as model:
        # Priors
        intercept = pm.Normal("intercept", mu=0, sigma=10)
        betas = pm.Normal("betas", mu=0, sigma=10, shape=degree)
        sigma = pm.HalfNormal("sigma", sigma=10)

        # Linear predictor
        mu = intercept + pm.math.dot(X_std, betas)

        # Likelihood
        obs = pm.Normal("obs", mu=mu, sigma=sigma, observed=y)

        # Sample
        idata = pm.sample(
            2000, chains=4, random_seed=random_seed,
            return_inferencedata=True,
            idata_kwargs={"log_likelihood": True},
        )

        # Posterior predictive
        pm.sample_posterior_predictive(idata, extend_inferencedata=True, random_seed=random_seed)

    return model, idata, (X_std, means, stds)

In [ ]:
if pm is not None:
    model_configs = {
        "Linear (deg 1)": 1,
        "Quadratic (deg 2)": 2,
        "Cubic (deg 3)": 3,
        "Degree 5": 5,
    }

    models = {}
    idatas = {}
    design_info = {}

    for name, degree in model_configs.items():
        print(f"\nFitting {name}...")
        model, idata, dinfo = fit_polynomial_model(x, y, degree, name)
        models[name] = model
        idatas[name] = idata
        design_info[name] = dinfo
        print(f"  Done. Divergences: {idata.sample_stats['diverging'].sum().values}")

    print("\nAll models fitted.")
else:
    print("PyMC not available -- skipping model fitting.")

### Quick diagnostics check

Before comparing models, we must verify that all four models have converged. A comparison based on non-converged posteriors is meaningless.

In [ ]:
if pm is not None and az is not None:
    print(f"{'Model':<22} {'Max R-hat':>10} {'Min ESS':>10} {'Divergences':>12}")
    print("-" * 56)

    for name, idata in idatas.items():
        rhat_vals = az.rhat(idata)
        max_rhat = max(rhat_vals[v].values.max() for v in rhat_vals.data_vars)

        ess_vals = az.ess(idata, method="bulk")
        min_ess = min(ess_vals[v].values.min() for v in ess_vals.data_vars)

        n_divs = int(idata.sample_stats["diverging"].sum().values)

        status = "OK" if (max_rhat < 1.01 and min_ess > 400 and n_divs == 0) else "CHECK"
        print(f"{name:<22} {max_rhat:10.4f} {min_ess:10.0f} {n_divs:12d}   [{status}]")
else:
    print("PyMC/ArviZ not available -- skipping.")

All models should show $\hat{R} < 1.01$, ESS $> 400$, and zero divergences. If any model fails these checks, its comparison results should be treated with caution.

---

## 7. Computing WAIC with ArviZ

ArviZ makes WAIC computation a one-liner. The function `az.waic()` takes an InferenceData object (which must contain the `log_likelihood` group) and returns the WAIC estimate, its standard error, and the effective number of parameters.

In [ ]:
if az is not None and pm is not None:
    print("WAIC for each model:\n")
    print(f"{'Model':<22} {'WAIC':>10} {'p_waic':>10} {'SE':>10}")
    print("-" * 54)

    waic_results = {}
    for name, idata in idatas.items():
        waic = az.waic(idata)
        waic_results[name] = waic
        print(f"{name:<22} {waic.waic:10.2f} {waic.p_waic:10.2f} {waic.waic_se:10.2f}")

    best_waic = min(waic_results, key=lambda k: waic_results[k].waic)
    print(f"\nBest model by WAIC: {best_waic}")
else:
    print("ArviZ not available -- skipping.")

**Reading the output:**

- **WAIC**: The WAIC value on the deviance scale ($-2 \times \widehat{\text{ELPD}}$). **Lower is better.**
- **p_waic**: The effective number of parameters. For well-identified models with weak priors, this is close to the actual parameter count. With informative priors, it can be smaller (the prior regularises).
- **SE**: Standard error of the WAIC estimate, computed from the pointwise contributions.

Notice how $p_{\text{WAIC}}$ grows with model complexity. The linear model has $\approx 3$ effective parameters (intercept, slope, $\sigma$), the quadratic $\approx 4$, and so on. Unlike the fixed count $p$ in AIC, $p_{\text{WAIC}}$ is data-adaptive.

---

## 8. Computing LOO-CV with ArviZ

Similarly, `az.loo()` computes the PSIS-LOO estimate of ELPD and reports the Pareto $\hat{k}$ diagnostics.

In [ ]:
if az is not None and pm is not None:
    print("LOO-CV (PSIS) for each model:\n")
    print(f"{'Model':<22} {'LOO':>10} {'p_loo':>10} {'SE':>10} {'bad k (>0.7)':>14}")
    print("-" * 68)

    loo_results = {}
    for name, idata in idatas.items():
        loo = az.loo(idata, pointwise=True)
        loo_results[name] = loo
        n_bad_k = np.sum(loo.pareto_k.values > 0.7)
        print(f"{name:<22} {loo.loo:10.2f} {loo.p_loo:10.2f} {loo.loo_se:10.2f} {n_bad_k:14d}")

    best_loo = min(loo_results, key=lambda k: loo_results[k].loo)
    print(f"\nBest model by LOO: {best_loo}")
else:
    print("ArviZ not available -- skipping.")

**Reading the output:**

- **LOO**: The LOO-IC value on the deviance scale. **Lower is better.**
- **p_loo**: Effective number of parameters (analogous to $p_{\text{WAIC}}$). If $p_{\text{loo}}$ is much larger than the actual parameter count, the model may be misspecified or overfitting.
- **SE**: Standard error of the LOO estimate.
- **bad k (>0.7)**: Number of observations with Pareto $\hat{k} > 0.7$. These are the observations where the importance sampling approximation is unreliable. Ideally this should be 0.

### Inspecting Pareto $\hat{k}$ values

In [ ]:
if az is not None and pm is not None:
    fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True, sharey=True)

    for ax, (name, loo) in zip(axes.flat, loo_results.items()):
        k_vals = loo.pareto_k.values
        colors = np.where(k_vals < 0.5, "C0",
                 np.where(k_vals < 0.7, "C1", "C3"))
        ax.scatter(range(len(k_vals)), k_vals, c=colors, s=15, alpha=0.7)
        ax.axhline(0.5, color="C1", linestyle="--", alpha=0.5, label="$\hat{k} = 0.5$")
        ax.axhline(0.7, color="C3", linestyle="--", alpha=0.5, label="$\hat{k} = 0.7$")
        ax.set_title(name, fontsize=11)
        ax.set_ylabel("Pareto $\hat{k}$")
        ax.set_xlabel("Observation index")
        ax.legend(fontsize=8)

    plt.suptitle("Pareto $\hat{k}$ Diagnostics for PSIS-LOO", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("ArviZ not available -- skipping.")

All Pareto $\hat{k}$ values should be below 0.7 (blue dots). If any observations show $\hat{k} > 0.7$ (red dots), the LOO estimate for those points is unreliable. For our well-specified models with moderately-sized data, we expect all diagnostics to be clean.

---

## 9. Comparing Models with `az.compare()`

The `az.compare()` function is the workhorse of Bayesian model comparison in ArviZ. It takes a dictionary of InferenceData objects, computes LOO (or WAIC), and produces a comparison table ranked by ELPD.

The key columns in the comparison table are:

| Column | Meaning |
|--------|----------|
| `rank` | Model rank (0 = best) |
| `elpd_loo` | Estimated ELPD from LOO |
| `p_loo` | Effective number of parameters |
| `d_loo` | Difference in ELPD from the best model |
| `weight` | Bayesian model averaging weight (pseudo-BMA or stacking) |
| `se` | Standard error of ELPD |
| `dse` | Standard error of the ELPD *difference* |
| `warning` | True if any Pareto $\hat{k} > 0.7$ |

In [ ]:
if az is not None and pm is not None:
    comparison = az.compare(idatas, ic="loo")
    print("Model Comparison (LOO):\n")
    print(comparison)
else:
    print("ArviZ not available -- skipping.")

### Interpreting the comparison table

The models are ranked by ELPD (highest = best). The critical column is `d_loo` -- the difference in ELPD from the best model. A model is meaningfully worse than the best if:

$$|d_{\text{loo}}| > 2 \times dse$$

That is, the ELPD difference is more than two standard errors away from zero. If the difference is small relative to its standard error, the models are effectively indistinguishable in terms of predictive performance.

The **weight** column gives stacking weights for Bayesian model averaging. A model with weight $\approx 1$ dominates; weights near zero indicate the model contributes little to the predictive mixture.

In [ ]:
if az is not None and pm is not None:
    az.plot_compare(comparison, figsize=(10, 3))
    plt.title("Model Comparison: ELPD Differences (LOO)", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("ArviZ not available -- skipping.")

The `az.plot_compare()` forest plot shows each model's ELPD with its standard error. The best model is marked with a filled circle. Open circles show the ELPD for other models. The horizontal bars are standard errors. Models whose error bars overlap with the best model cannot be reliably distinguished.

We can also run the comparison using WAIC instead of LOO:

In [ ]:
if az is not None and pm is not None:
    comparison_waic = az.compare(idatas, ic="waic")
    print("Model Comparison (WAIC):\n")
    print(comparison_waic)
else:
    print("ArviZ not available -- skipping.")

In practice, LOO and WAIC give very similar rankings. LOO (with PSIS) is generally preferred because:
1. It has a built-in diagnostic (Pareto $\hat{k}$) that tells you when to be worried.
2. It is more robust for models with weak or misspecified likelihoods.

The ArviZ documentation recommends LOO as the default.

---

## 10. Bayes Factors (Brief)

### Definition

An alternative approach to Bayesian model comparison is the **Bayes factor**. Given two models $M_1$ and $M_2$, the Bayes factor is:

$$B_{10} = \frac{p(y \mid M_1)}{p(y \mid M_2)}$$

where $p(y \mid M_k)$ is the **marginal likelihood** (or **evidence**) of model $k$:

$$p(y \mid M_k) = \int p(y \mid \theta_k, M_k)\, p(\theta_k \mid M_k)\, d\theta_k$$

This is the likelihood of the data averaged over the prior distribution of the parameters. It is precisely the normalising constant in Bayes' theorem that we usually ignore.

### Jeffreys' interpretation scale

| $B_{10}$ | $\log_{10} B_{10}$ | Evidence for $M_1$ |
|---------|-------------------|---------------------|
| $1 - 3$ | $0 - 0.5$ | Barely worth mentioning |
| $3 - 10$ | $0.5 - 1$ | Substantial |
| $10 - 30$ | $1 - 1.5$ | Strong |
| $30 - 100$ | $1.5 - 2$ | Very strong |
| $> 100$ | $> 2$ | Decisive |

### Connection to posterior model probabilities

If we assign prior model probabilities $P(M_1)$ and $P(M_2) = 1 - P(M_1)$, the posterior odds are:

$$\frac{P(M_1 \mid y)}{P(M_2 \mid y)} = B_{10} \cdot \frac{P(M_1)}{P(M_2)}$$

With equal prior odds ($P(M_1) = P(M_2) = 0.5$), the posterior odds equal the Bayes factor.

### Why Bayes factors are problematic in practice

Despite their elegant theoretical foundation, Bayes factors have serious practical limitations:

1. **The marginal likelihood is hard to compute.** It is an integral over the entire parameter space. MCMC gives us samples from the posterior, not the normalising constant. Specialised algorithms exist (bridge sampling, nested sampling, thermodynamic integration) but they are computationally expensive and tricky to get right.

2. **Extreme sensitivity to priors.** The marginal likelihood depends strongly on the prior, even for vague priors. Consider: if you use $\beta \sim \mathcal{N}(0, 100)$, the prior spreads probability over a huge volume of parameter space. The marginal likelihood penalises this "wasted" prior mass. Switching to $\beta \sim \mathcal{N}(0, 10)$ can change the Bayes factor by orders of magnitude, even though both priors are "non-informative" in the usual sense.

   This is not a bug -- it reflects the Bayesian principle that models which make vague predictions are penalised. But it means that Bayes factors for models with diffuse priors are essentially arbitrary.

3. **Improper priors are forbidden.** If you use an improper prior (e.g., flat over $(-\infty, \infty)$), the marginal likelihood is infinite or undefined, and the Bayes factor is meaningless.

4. **No diagnostic for reliability.** Unlike LOO (with Pareto $\hat{k}$), Bayes factors give you a number with no built-in indication of whether that number is trustworthy.

### The recommendation

For most applied work, **LOO-CV (via PSIS) is preferred over Bayes factors.** LOO directly estimates predictive accuracy, is robust to prior choices, works with improper priors, and has a built-in diagnostic. Use Bayes factors when:
- You have genuine, carefully specified prior distributions (not just "default" priors)
- The scientific question is genuinely about which model is "true" (rare)
- You are comparing nested models with simple structure

---

## 11. Frequentist Comparison: AIC and BIC

For completeness, let us compute AIC and BIC from Module 06 on the same data, using `statsmodels`. This lets us see whether the frequentist and Bayesian criteria agree.

In [ ]:
import statsmodels.api as sm

freq_results = {}
degrees_to_compare = [1, 2, 3, 5]

print(f"{'Model':<22} {'AIC':>10} {'BIC':>10}")
print("-" * 44)

for deg in degrees_to_compare:
    X_poly = np.column_stack([x**j for j in range(1, deg + 1)])
    X_poly = sm.add_constant(X_poly)
    ols_model = sm.OLS(y, X_poly).fit()
    name = f"Degree {deg}"
    freq_results[name] = {"aic": ols_model.aic, "bic": ols_model.bic}
    print(f"{name:<22} {ols_model.aic:10.2f} {ols_model.bic:10.2f}")

best_aic = min(freq_results, key=lambda k: freq_results[k]["aic"])
best_bic = min(freq_results, key=lambda k: freq_results[k]["bic"])
print(f"\nBest by AIC: {best_aic}")
print(f"Best by BIC: {best_bic}")

In [ ]:
if az is not None and pm is not None:
    # --- Summary comparison: Bayesian vs. Frequentist ---
    model_names = ["Linear (deg 1)", "Quadratic (deg 2)", "Cubic (deg 3)", "Degree 5"]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Panel 1: Bayesian (LOO and WAIC)
    ax = axes[0]
    loo_vals = [loo_results[name].loo for name in model_names]
    waic_vals = [waic_results[name].waic for name in model_names]
    x_pos = np.arange(len(model_names))

    ax.bar(x_pos - 0.15, loo_vals, width=0.3, label="LOO-IC", alpha=0.8)
    ax.bar(x_pos + 0.15, waic_vals, width=0.3, label="WAIC", alpha=0.8)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(["Deg 1", "Deg 2", "Deg 3", "Deg 5"], fontsize=10)
    ax.set_ylabel("Information criterion (lower is better)")
    ax.set_title("Bayesian: LOO-IC and WAIC")
    ax.legend()

    # Panel 2: Frequentist (AIC and BIC)
    ax = axes[1]
    freq_names = [f"Degree {d}" for d in degrees_to_compare]
    aic_vals = [freq_results[name]["aic"] for name in freq_names]
    bic_vals = [freq_results[name]["bic"] for name in freq_names]

    ax.bar(x_pos - 0.15, aic_vals, width=0.3, label="AIC", alpha=0.8, color="C2")
    ax.bar(x_pos + 0.15, bic_vals, width=0.3, label="BIC", alpha=0.8, color="C3")
    ax.set_xticks(x_pos)
    ax.set_xticklabels(["Deg 1", "Deg 2", "Deg 3", "Deg 5"], fontsize=10)
    ax.set_ylabel("Information criterion (lower is better)")
    ax.set_title("Frequentist: AIC and BIC")
    ax.legend()

    plt.suptitle("Bayesian vs. Frequentist Model Selection", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("ArviZ not available -- skipping.")

All four criteria -- LOO, WAIC, AIC, and BIC -- should agree in selecting the quadratic model as best. This is reassuring: when the data clearly favours one model, Bayesian and frequentist approaches converge. The Bayesian criteria offer additional value when models are close in fit, through the standard error of the ELPD difference and the Pareto $\hat{k}$ diagnostic.

---

## 12. Posterior Predictive Comparison

Beyond information criteria, we should **visually inspect** how each model's posterior predictive distribution matches the data. A model might have a good ELPD but still show systematic misfit in certain regions.

For each model, we generate posterior predictive samples at a grid of $x$ values and plot the mean prediction with uncertainty bands.

In [ ]:
if pm is not None and az is not None:

    def posterior_predictive_at_grid(idata, x_new, degree, design_info_tuple):
        """
        Compute posterior predictive mean and std at new x values.
        """
        _, means, stds = design_info_tuple
        X_new = np.column_stack([x_new**j for j in range(1, degree + 1)])
        X_new_std = (X_new - means) / stds

        # Extract posterior samples
        intercept = idata.posterior["intercept"].values.flatten()
        betas = idata.posterior["betas"].values.reshape(-1, degree)
        sigma = idata.posterior["sigma"].values.flatten()

        # Compute predicted mean for each posterior draw
        # Shape: (n_draws, n_grid)
        mu_pred = intercept[:, None] + betas @ X_new_std.T

        return mu_pred, sigma

    fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
    model_names = ["Linear (deg 1)", "Quadratic (deg 2)", "Cubic (deg 3)", "Degree 5"]
    model_degrees = [1, 2, 3, 5]

    for ax, name, deg in zip(axes.flat, model_names, model_degrees):
        idata = idatas[name]
        dinfo = design_info[name]

        mu_pred, sigma_samples = posterior_predictive_at_grid(
            idata, x_grid, deg, dinfo
        )

        # Posterior predictive mean and intervals
        mean_pred = mu_pred.mean(axis=0)
        # Credible interval for the mean function
        ci_low = np.percentile(mu_pred, 2.5, axis=0)
        ci_high = np.percentile(mu_pred, 97.5, axis=0)
        # Prediction interval (includes noise)
        sigma_mean = sigma_samples.mean()
        pi_low = np.percentile(mu_pred, 2.5, axis=0) - 2 * sigma_mean
        pi_high = np.percentile(mu_pred, 97.5, axis=0) + 2 * sigma_mean

        ax.scatter(x, y, s=15, alpha=0.5, color="black", zorder=5)
        ax.plot(x_grid, true_function(x_grid), "k--", linewidth=1.5, alpha=0.4,
                label="True function")
        ax.plot(x_grid, mean_pred, color="C0", linewidth=2, label="Posterior mean")
        ax.fill_between(x_grid, ci_low, ci_high, alpha=0.3, color="C0",
                        label="95% CI (mean)")
        ax.fill_between(x_grid, pi_low, pi_high, alpha=0.1, color="C0",
                        label="~95% PI (obs)")

        # Get LOO for title
        loo_val = loo_results[name].loo
        ax.set_title(f"{name} | LOO-IC = {loo_val:.1f}", fontsize=11)
        ax.legend(fontsize=8, loc="upper right")

    axes[1, 0].set_xlabel("$x$")
    axes[1, 1].set_xlabel("$x$")
    axes[0, 0].set_ylabel("$y$")
    axes[1, 0].set_ylabel("$y$")

    plt.suptitle("Posterior Predictive Comparison Across Models", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("PyMC/ArviZ not available -- skipping.")

**What to look for:**

- **Linear model (underfitting):** The mean prediction is a straight line -- it misses the curvature entirely. The prediction intervals are wide because the model compensates for systematic misfit with extra noise.

- **Quadratic model (just right):** The posterior mean closely tracks the true function. The credible intervals are narrow and the prediction intervals are reasonable.

- **Cubic model (slightly overfit):** Very similar to the quadratic, because the extra cubic term gets a posterior concentrated near zero. The credible intervals may be slightly wider.

- **Degree-5 model (overfit):** The mean prediction may wiggle near the boundaries. The credible intervals are wider, especially at the edges, reflecting the increased parameter uncertainty. The prediction intervals may be excessively wide.

This visual comparison reinforces the LOO/WAIC ranking: the quadratic model provides the best balance of fit and parsimony.

### Posterior predictive checks with `az.plot_ppc()`

We can also use ArviZ's posterior predictive check to compare the distribution of observed data against simulated data from each model.

In [ ]:
if az is not None and pm is not None:
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    for ax, (name, idata) in zip(axes.flat, idatas.items()):
        az.plot_ppc(idata, num_pp_samples=100, ax=ax, legend=False)
        ax.set_title(name, fontsize=11)

    # Add legend to first panel only
    axes[0, 0].legend(fontsize=8)

    plt.suptitle("Posterior Predictive Checks: Observed vs. Simulated Data",
                 fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("ArviZ not available -- skipping.")

In the PPC plots, the thin blue lines are simulated replications from the posterior predictive, and the dark line is the observed data distribution. For well-specified models (quadratic and above), the simulated replications should closely envelope the observed data.

---

## 13. Summary: A Model Comparison Workflow

Putting it all together, here is a recommended workflow for Bayesian model comparison:

```
1. Fit candidate models in PyMC
   └── Include log_likelihood=True in idata_kwargs

2. Check MCMC diagnostics for ALL models
   ├── R-hat < 1.01
   ├── ESS > 400 (bulk and tail)
   └── No divergent transitions

3. Compute LOO with az.loo() for each model
   └── Check Pareto k diagnostics (all < 0.7)

4. Compare with az.compare()
   ├── Rank by ELPD
   ├── Check if differences are > 2*dse
   └── Examine stacking weights

5. Visualise with az.plot_compare()
   └── Forest plot of ELPD differences

6. Posterior predictive checks
   ├── az.plot_ppc() for distributional fit
   └── Plot predictions vs. data across the predictor range

7. Make a decision
   ├── Best ELPD model if clearly separated
   ├── Simpler model if ELPD difference < 2*dse (parsimony)
   └── Model averaging if multiple models contribute (stacking weights)
```

---

## Exercises

**Exercise 3.1 (WAIC vs. LOO).** Using the four fitted models from this notebook, compare the WAIC and LOO estimates of ELPD side by side. (a) Make a scatter plot of $\widehat{\text{ELPD}}_{\text{WAIC}}$ vs. $\widehat{\text{ELPD}}_{\text{LOO}}$ for all four models. How closely do they agree? (b) Compute the *pointwise* differences between WAIC and LOO contributions for each observation in the quadratic model. Which observations show the largest discrepancy? Do these correspond to influential points?

**Exercise 3.2 (Prior sensitivity of LOO).** Refit the quadratic model with three different prior scales for the coefficients: $\beta_j \sim \mathcal{N}(0, \sigma_\beta)$ with $\sigma_\beta \in \{1, 10, 100\}$. Compute LOO for each. How sensitive is the LOO-ELPD to the prior choice? Now compute the marginal likelihood (use `pm.compute_log_likelihood` or bridge sampling if available) and compare how sensitive the Bayes factor is. This illustrates the key advantage of LOO over Bayes factors.

**Exercise 3.3 (Pareto $\hat{k}$ investigation).** Add 3 outliers to the dataset: observations with $y$ values 5 standard deviations from the mean. Refit the quadratic model and compute LOO. (a) Which observations have $\hat{k} > 0.7$? (b) Plot the Pareto $\hat{k}$ values and colour the outliers differently. (c) What does this tell you about the relationship between influential observations and the LOO diagnostic?

**Exercise 3.4 (Model averaging).** Use the stacking weights from `az.compare()` to construct a weighted posterior predictive distribution. For each $x$ value on a grid: sample from each model's posterior predictive with probability equal to its stacking weight, then combine. Plot the averaged predictions against the individual model predictions. Does the averaged prediction outperform the best single model?

**Exercise 3.5 (Simulation study).** Repeat the full model comparison workflow 50 times, each time simulating a new dataset from the quadratic true model. In what fraction of simulations does LOO correctly identify the quadratic model as best (rank 1)? In what fraction is the quadratic within the top 2? How does this compare with AIC and BIC?

---

## Key Takeaways

1. **ELPD is the fundamental quantity** for Bayesian model comparison. Higher ELPD means better out-of-sample predictive accuracy. Both WAIC and LOO estimate ELPD.

2. **WAIC** = lppd $-$ $p_{\text{WAIC}}$. It generalises AIC to the full Bayesian setting, using the posterior variance of the log-likelihood as the complexity penalty. Computed from existing posterior samples -- no refitting.

3. **PSIS-LOO** estimates leave-one-out cross-validation via importance sampling, stabilised by Pareto smoothing. It is the recommended default in ArviZ because of its built-in Pareto $\hat{k}$ diagnostic.

4. **Pareto $\hat{k}$** tells you when to trust LOO. Values $> 0.7$ signal unreliable estimates for specific observations. Always check this diagnostic.

5. **`az.compare()`** ranks models by ELPD, reports standard errors of differences, and provides stacking weights for model averaging. A model is meaningfully worse if $|d_{\text{loo}}| > 2 \times dse$.

6. **Bayes factors** compare marginal likelihoods and have a clean theoretical foundation, but they are computationally hard, extremely prior-sensitive, and lack a reliability diagnostic. For most applied work, LOO is preferred.

7. **Always combine quantitative criteria with visual checks.** Posterior predictive plots reveal systematic misfit that summary statistics can miss.

---

This completes **Module 08: Bayesian Regression**. We have covered Bayesian linear regression, posterior predictive distributions, and now model comparison.

**Next module:** [Module 09 -- Hierarchical Models](../09_hierarchical_models/01_partial_pooling.ipynb): We extend Bayesian regression to hierarchical (multilevel) models, where parameters are shared across groups through partial pooling.